# 实践 12.2  多环境预测的随机记录验证与留一环境验证

《统计遗传与智能育种学》第十二章 · 电子资源 `12b_gs_gxe_validation.ipynb`

本 notebook 的代码与书中“完整Python代码”逐字一致。运行检查：重启内核并运行全部单元（Restart Kernel and Run All）。

电子资源：12b_gs_gxe_validation.ipynb。实践生成240个基因型、连续梯度上的6个环境和已知环境敏感度，使表型同时包含加性主效应、环境主效应与G×E。代码比较G-only、G+E和G+E+G×E三种岭回归模型；所有标准化和系数都只在训练记录拟合。评价同时报告相关r、RMSE和平均偏差，检验单一留出环境内的排序、误差大小和均值偏差。留出e=2.0时所有基因型都在其他环境出现过，因此该场景是已见基因型+未见极端环境，不代表新基因型预测。

运行检查：重启内核并运行全部单元  
本书验证环境实际结果：随机记录划分中，G/G+E/G+E+G×E的r分别为-0.025/0.883/0.896，RMSE为2.270/0.993/0.942；完整留出极端环境e=2.0时，三者r为0.413/0.413/0.620，而RMSE为3.218/1.061/0.943，G-only平均偏差为-3.038。利用全部模拟记录计算的基因型均值、环境斜率与相应真值的相关分别为0.882、0.760。结果说明相关、误差尺度和偏差必须联合报告。

## 输入、完整代码、实际输出与解释

输入与可复现条件：240个基因型×6个环境、180个标记；SEED=1202；环境梯度[-2,-1.2,-0.4,0.4,1.2,2]；模拟真值含基因型主效应、斜率为1.3的环境主效应和基因型特异环境斜率。

In [1]:
import sys, platform
import numpy as np
import pandas as pd
import sklearn
SEED = 1202
rng = np.random.default_rng(SEED)
print('Python', sys.version.split()[0])
print('Platform', platform.platform())
print('numpy', np.__version__, 'pandas', pd.__version__, 'scikit-learn', sklearn.__version__)
print('SEED', SEED)


Python 3.12.13
Platform macOS-26.5.2-arm64-arm-64bit
numpy 2.3.5 pandas 2.2.3 scikit-learn 1.8.0
SEED 1202


In [2]:
n_g, n_e, m = 240, 6, 180
p = rng.uniform(0.1, 0.5, m)
Xg = rng.binomial(2, p, size=(n_g, m)).astype(float)
Xc = Xg - Xg.mean(axis=0)
beta = rng.normal(0, 1/np.sqrt(m), m)
gamma = rng.normal(0, 1/np.sqrt(m), m)
g = Xc @ beta
sens = (Xc @ gamma)/2
env = np.linspace(-2, 2, n_e)
rows = []
for i in range(n_g):
    for j, e in enumerate(env):
        y = g[i] + 1.3*e + sens[i]*e + rng.normal(0, 0.8)
        rows.append((i, j, e, y))
df = pd.DataFrame(rows, columns=['geno','env_id','env','y'])
print('environment gradient', env)
print(df.head())


environment gradient [-2.  -1.2 -0.4  0.4  1.2  2. ]
   geno  env_id  env         y
0     0       0 -2.0 -1.335085
1     0       1 -1.2 -1.610656
2     0       2 -0.4 -0.335825
3     0       3  0.4  0.279783
4     0       4  1.2  0.780009


In [3]:
from sklearn.linear_model import Ridge
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline

def features(frame, model_kind):
    G = Xg[frame.geno.to_numpy()]
    e = frame.env.to_numpy()[:,None]
    if model_kind == 'G':
        return G
    if model_kind == 'G+E':
        return np.hstack([G, e])
    if model_kind == 'G+E+GxE':
        return np.hstack([G, e, G*e])
    raise ValueError(model_kind)

def fit_eval(train_idx, test_idx, model_kind):
    model = make_pipeline(StandardScaler(), Ridge(alpha=20))
    model.fit(features(df.iloc[train_idx], model_kind), df.y.iloc[train_idx])
    pred = model.predict(features(df.iloc[test_idx], model_kind))
    obs = df.y.iloc[test_idx].to_numpy()
    return {
        'model': model_kind,
        'r': float(np.corrcoef(pred, obs)[0,1]),
        'RMSE': float(np.sqrt(np.mean((pred-obs)**2))),
        'bias': float(np.mean(pred-obs))
    }

all_idx = np.arange(len(df))
rng.shuffle(all_idx)
cut = int(0.8*len(df))
tr_random, te_random = all_idx[:cut], all_idx[cut:]
hold_env = n_e-1
tr_env = np.where(df.env_id.to_numpy()!=hold_env)[0]
te_env = np.where(df.env_id.to_numpy()==hold_env)[0]

rows_eval = []
for scenario, tr, te in [
    ('random-record', tr_random, te_random),
    ('leave-env-out e=2.0', tr_env, te_env)
]:
    for kind in ['G','G+E','G+E+GxE']:
        row = fit_eval(tr, te, kind)
        row['scenario'] = scenario
        rows_eval.append(row)
print(pd.DataFrame(rows_eval)[['scenario','model','r','RMSE','bias']].round(3).to_string(index=False))


           scenario   model      r  RMSE   bias
      random-record       G -0.025 2.270  0.014
      random-record     G+E  0.883 0.993  0.006
      random-record G+E+GxE  0.896 0.942  0.110
leave-env-out e=2.0       G  0.413 3.218 -3.038
leave-env-out e=2.0     G+E  0.413 1.061 -0.048
leave-env-out e=2.0 G+E+GxE  0.620 0.943 -0.009


In [4]:
geno_mean = df.groupby('geno').y.mean().to_numpy()
slope = np.array([
    np.polyfit(df.loc[df.geno==i,'env'], df.loc[df.geno==i,'y'], 1)[0]
    for i in range(n_g)
])
print('recovery of genetic main effect r', round(float(np.corrcoef(geno_mean,g)[0,1]),3))
print('recovery of environment sensitivity r',
      round(float(np.corrcoef(slope-1.3,sens)[0,1]),3))


recovery of genetic main effect r 0.882
recovery of environment sensitivity r 0.76


结果解释：留出e=2.0时，G-only与G+E的相关都为0.413，本例训练设计平衡，加入环境效应只给留出环境的预测加同一常数，相关不变；但G-only的RMSE=3.218、bias=-3.038，表明平均预测值偏低。G+E减小均值偏差，G+E+G×E与观测表型的相关更高，使r升到0.620。这里用标记与环境协变量的乘积构建岭回归，作为多环境预测的简化示例。

可修改练习：把hold_env改为3（e=0.4，处于训练环境范围内部）与极端e=2.0比较；双外推时，训练同时排除留出基因型和留出环境的全部记录，测试取两者交集。